Train models with only the most important feature

In [ ]:
estimator = {
    "name": "XGBClassifier",
    "steps": [
        {
            "name": "XGBClassifier",
            "args": {
                "max_depth": [3, 5, 9],
                "min_child_weight": [1, 5, 10, 50],
                "n_jobs": [-1],
                "n_estimators": [10, 50, 100, 200],
                "learning_rate": [0.01, 0.05, 0.1, 0.2],
                "max_bin": 16,
                "objective": ["binary:logistic"],
                "tree_method": ["hist"],
            }
        }
    ]
}

In [9]:
import pandas as pd
import numpy as np
import os

from imblearn.over_sampling import RandomOverSampler
from imblearn.pipeline import Pipeline

from xgboost import XGBClassifier

def build_pipeline(estimator, random_state):
    pipeline_steps = []
    for i, step in enumerate(estimator["steps"]):
        step_name = step["name"]
        step_args = step.get("args", {})  # get the hyperparameters for the step, if any
        step_args.update({"random_state": random_state})
        step_instance = globals()[step_name](
            **step_args
        )  # dynmically create an instance of the step
        pipeline_steps.append((step_name, step_instance))

    # Oversampling train set
    pipeline_steps.insert(-1, ("oversample", RandomOverSampler(random_state=random_state)))

    pipeline = Pipeline(pipeline_steps)

    return pipeline


def build_param_grid(estimator_steps):
    param_grid = {}
    for step in estimator_steps:
        step_name = step["name"]
        step_args = step.get("args", {})
        param_grid.update(
            {
                f"{step_name}__{key}": value
                for key, value in step_args.items()
                if isinstance(value, list)
            }
        )
    return param_grid


def perform_grid_search_cv(
    X_train, y_train, estimator, pipeline, n_splits, n_repeats, random_state
):
    from sklearn.model_selection import GridSearchCV, RepeatedStratifiedKFold

    cv = RepeatedStratifiedKFold(
        n_splits=n_splits,
        n_repeats=n_repeats,
        random_state=random_state,
    )

    grid_search_cv = GridSearchCV(
        pipeline,
        param_grid=build_param_grid(estimator["steps"]),
        cv=cv,
        scoring="balanced_accuracy",
        n_jobs=-1,
        verbose=0,
    )

    # Use train set as input to Grid Search Cross Validation
    # (k-fold cross-validation sets drawn internally from train set)
    grid_search_cv = grid_search_cv.fit(X_train, y_train)

    best_params = (
        grid_search_cv.best_params_ if grid_search_cv.best_params_ else "default"
    )

    return grid_search_cv

def save_parameters(results_parameters, output_dir, aeid):
    results_parameter_path = os.path.join(output_dir, f'{aeid}_grid_search.csv')
    results_parameters.to_csv(results_parameter_path, index= False)


In [17]:
from matplotlib import pyplot as plt
def predict_and_report_classification(
    X,
    y,
    best_estimator,
    default_threshold,
    output_dir,
    aeid,
    cmap="Blues",
):
    from sklearn.metrics import (
        precision_score,
        recall_score,
        f1_score,
        balanced_accuracy_score,
        average_precision_score,
        roc_auc_score,
        matthews_corrcoef,
        confusion_matrix,
        ConfusionMatrixDisplay,
    )

    # Get the prediction probabilites
    y_pred_proba = best_estimator.predict_proba(X)[:, 1]
    print("Mean predicted probability:", y_pred_proba.mean())
    print("Range:", (y_pred_proba.min(), y_pred_proba.max()))

    # Get the predictions for the default threshold

    y_pred_default_threshold = np.where(y_pred_proba >= default_threshold, 1, 0)

    # Get common metrics
    support = len(y)
    n_pos = np.sum(y)
    f1_micro = f1_score(y, y_pred_default_threshold, average="micro")
    f1_macro = f1_score(y, y_pred_default_threshold, average="macro")
    recall_micro = recall_score(y, y_pred_default_threshold, average="micro")
    recall_macro = recall_score(y, y_pred_default_threshold, average="macro")
    precision_micro = precision_score(y, y_pred_default_threshold, average="micro")
    precision_macro = precision_score(y, y_pred_default_threshold, average="macro")
    balanced_accuracy = balanced_accuracy_score(y, y_pred_default_threshold)
    mcc = matthews_corrcoef(y, y_pred_default_threshold)
    pr_auc = average_precision_score(y, y_pred_proba)
    roc_auc = roc_auc_score(y, y_pred_proba)

    # Save the metrics
    metrics_df = pd.DataFrame(
        {
            "support": [support],
            "n_pos": [n_pos],
            "recall_micro": [recall_micro],
            "recall_macro": [recall_macro],
            "precision_micro": [precision_micro],
            "precision_macro": [precision_macro],
            "f1_micro": [f1_micro],
            "f1_macro": [f1_macro],
            "balanced_accuracy": [balanced_accuracy],
            "mcc": [mcc],
            "pr_auc": [pr_auc],
            "roc_auc": [roc_auc],
        }
    )
    metrics_df_path = os.path.join(output_dir, f'{aeid}_metrics.csv')
    metrics_df.round(5).to_csv(metrics_df_path, index=False)

    labels = [False, True]

    cm = confusion_matrix(
            y,
            y_pred_default_threshold,
            labels=labels,
        )

    tn, fp, fn, tp = cm.ravel()  # Extract values from confusion matrix


    display_labels = {
        "Negative": {"fontsize": 30},
        "Positive": {"fontsize": 30},
    }

    plt.figure(figsize=(8, 8))
    cm_display = ConfusionMatrixDisplay(
                    confusion_matrix=cm, display_labels=display_labels
    )
    cm_display.plot(cmap=cmap, colorbar=False)

    pos_count = cm[1, 0] + cm[1, 1]
    neg_count = cm[0, 0] + cm[0, 1]

    plt.title(
        f"aeid: {aeid}, Count: {len(y)} (P:{pos_count}, N:{neg_count})",
        fontsize=10,
    )
    fig_path = os.path.join(output_dir, f'{aeid}_cm.svg')
    plt.savefig(fig_path, format="svg")
    plt.close("all")


In [5]:
pytcpl_dir = r"Q:\Abteilungsprojekte\uchem\Projekte\Aktuelle Projekte\1179_MLinvitroTox\MLinvitroTox_PosNeg\mlinvitrotox\data\input\invitro\pytcpl"
aeid = 980
hitcall_path = os.path.join(pytcpl_dir, f'{aeid}.parquet.gzip')
hitcall = pd.read_parquet(hitcall_path)
print(hitcall.head(3))

          ac50       ac95        acc  actop  bmd  ac1sd  aeid best_aic_model  \
48   31.061697  44.872966  33.682484  100.0  NaN    NaN   980           hill   
355        NaN        NaN        NaN    NaN  NaN    NaN   980           hill   
65    0.688413   7.354144        NaN   10.0  NaN    NaN   980           hill   

                                                  conc  \
48   [0.03, 0.09999999999999999, 3.0, 3.0, 100.0, 2...   
355  [100.0, 0.03, 10.0, 3.0, 0.03, 0.0999999999999...   
65   [29.999999999999996, 1.0, 29.999999999999996, ...   

                                            fit_params  ...       top  \
48   {"cnst": {"pars": {"a": 1.0, "er": -1.99387423...  ... -1.164409   
355  {"cnst": {"pars": {"a": 1.0, "er": -1.32739702...  ...       NaN   
65   {"cnst": {"pars": {"a": 1.0, "er": -1.97710376...  ...  0.317140   

     dsstox_substance_id                              cautionary_flags fitc  \
48         DTXSID0020022  ["Only highest conc above baseline, active"]   4

In [19]:

aeids = [990, 1058, 1066]
feature = "714"
root_dir = r"Q:\Abteilungsprojekte\uchem\Projekte\Aktuelle Projekte\1179_MLinvitroTox\MLinvitroTox_PosNeg\mlinvitrotox\data\results\models\final_models\class"
toxcast_true_fps= pd.read_csv(r"Q:\Abteilungsprojekte\uchem\Projekte\Aktuelle Projekte\1179_MLinvitroTox\MLinvitroTox_PosNeg\mlinvitrotox\data\processed\chemical_fingerprints\ToxCast_True_FPS.csv")
pytcpl_dir = r"Q:\Abteilungsprojekte\uchem\Projekte\Aktuelle Projekte\1179_MLinvitroTox\MLinvitroTox_PosNeg\mlinvitrotox\data\input\invitro\pytcpl"
output_dir = r"C:\Users\aischlen\Documents"

for aeid in aeids:

    aeid_path = os.path.join(root_dir, f"{aeid}")
    selected_chem_path = os.path.join(aeid_path, "selected_chemicals.csv")
    test_true_chem_path = os.path.join(aeid_path, "test_true_chemicals.csv")

    hitcall_path = os.path.join(pytcpl_dir, f'{aeid}.parquet.gzip')
    hitcall = pd.read_parquet(hitcall_path)

    test_ids = pd.read_csv(test_true_chem_path)["0"]
    selected_ids = pd.read_csv(selected_chem_path)["DTXSID"]

    train_ids = selected_ids[~selected_ids.isin(test_ids)]

    data = toxcast_true_fps[
        ["DTXSID", feature]
    ].merge(
        hitcall[
            ["dsstox_substance_id", "hitcall_c"]
        ],
        left_on="DTXSID",
        right_on="dsstox_substance_id"
    )

    data = data.drop_duplicates(subset="DTXSID")

    train_data = data[data["DTXSID"].isin(train_ids)]
    test_data = data[data["DTXSID"].isin(test_ids)]

    train_data = train_data.copy()
    test_data = test_data.copy()

    train_data["hitcall_c"] = (train_data["hitcall_c"] >= 0.9).astype(int)
    test_data["hitcall_c"] = (test_data["hitcall_c"] >= 0.9).astype(int)

    X_train = train_data[[feature]]
    y_train = train_data["hitcall_c"]

    X_test = test_data[[feature]]
    y_test = test_data["hitcall_c"]

    ## Training
    # Build the pipeline for the current estimator with the specified parameter grid
    random_state = 42
    pipeline = build_pipeline(estimator, random_state)

    # Perform grid search cross-validation
    # (Note: CV on TRAINING set with RepeatedStratifiedKFold)

        
    n_splits = 5
    n_repeats = 3
    grid_search = perform_grid_search_cv(
        X_train, 
        y_train,
        estimator,
        pipeline,
        n_splits,
        n_repeats,
        random_state,
    )


    # the parameters
    best_params = grid_search.best_params_
    results_parameter = pd.DataFrame(grid_search.cv_results_)
    save_parameters(results_parameter, output_dir, aeid)

    # Save best estimator (estimator with best performing parameters from grid search)
    best_estimator = grid_search.best_estimator_


    # Predict on the 1. "true Massbank" and 2. "SIRIUS predicted" validation set
    default_threshold = 0.5

    predict_and_report_classification(
        X_test,
        y_test,
        best_estimator,
        default_threshold,
        output_dir,
        aeid
    )


Mean predicted probability: 0.49584436
Range: (np.float32(0.4792269), np.float32(0.52187836))
Mean predicted probability: 0.4970918
Range: (np.float32(0.48443902), np.float32(0.5179691))
Mean predicted probability: 0.49626222
Range: (np.float32(0.4847874), np.float32(0.5176369))
